### retrival testing

1. recall score 

In [ ]:

import re
import contractions
import emoji
from textblob import TextBlob
import spacy
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:


with open("data.txt", "r", encoding="utf-8") as file:
    data = file.read()

# Lowercase
data = data.lower()

# Remove extra spaces
data = re.sub(r"\s{2,}", " ", data)

# Expand contractions
data = contractions.fix(data)

# Remove emojis
data = emoji.replace_emoji(data, "")

# Remove punctuation
data = re.sub(r"[^a-z0-9\s]", "", data)

# Spell correction (optional)
# data = str(TextBlob(data).correct())

print(data)


machine learning is a branch of artificial intelligence that focuses on enabling computers to learn patterns from data
instead of explicitly programming every rule machine learning algorithms learn from examples and use those examples to make predictions
machine learning has become an important technology in many industries
it is used in healthcare finance education transportation retail entertainment and cybersecurity
the basic idea behind machine learning is simple
we provide data to an algorithm allow the algorithm to learn patterns and then use the trained model to make predictions on new data
machine learning can be divided into several major categories
the most common categories are supervised learning unsupervised learning semisupervised learning and reinforcement learning
each type of learning solves a different kind of problem
the choice of learning method depends on the type of data available and the objective of the project
supervised learning is a type of machine learning w

In [ ]:
nlp=spacy.load('en_core_web_sm')   #It is SpaCy's small English language model (for grammer training)
tokens=nlp(data)   #This line sends the sentence into SpaCy.

print(type(tokens))  #op tional
'''A Doc object is a container that stores the processed sentence '''

update_tokens=[token.lemma_ for token in tokens if not token.is_stop]  #token.lemma_ → human-readable text
data=' '.join(update_tokens).strip()  # we use strip to extra spaces

<class 'spacy.tokens.doc.Doc'>


In [ ]:
splitter=RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=40
)

# Generate chunks from the lemmatized text
'''chunks = splitter.split_text(data)'''  # it return all chunks
chunks = splitter.create_documents([data])


In [ ]:
embedding_model=HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-miniLM-L6-v2'  #to convert the chunks into vector
) 
# Every sentence becomes a list of numbers

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
vectordb=FAISS.from_documents(documents=chunks,embedding=embedding_model)
vectordb

In [ ]:
def r_search(query,k=5):    
    R_chunks =vectordb.similarity_search(query,k=k)
    return R_chunks

user_prompt = 'Explain Machine Learning ?'
user_prompt = re.sub(r'[^0-9a-zA-Z\s]','',user_prompt) # text normalization
R_chunks=r_search(user_prompt)

In [ ]:
parser=StrOutputParser()

In [ ]:
import os

In [ ]:
llm=ChatGoogleGenerativeAI(
    model='gemini-3.5-flash',
    api_key=os.environ['GEMINI_API_KEY'],
        temperature=0.2
    )

In [ ]:
Retrival_check=ChatPromptTemplate.from_messages([
    ('system',
     "You're a helpful assistant."
     "Detect if the {R_chunks} is relevant answer for the {user_prompt}"
     "Check the accuracy of the {R_chunks}"
     "Example: Question:what is ML?,out of 5 retrived if chunks 4 is correct and return the score 80"
     "If any chunk is wrong/not relevant,detect it and mention why it is wrong"
     ),
     ('human',"Retrived chunk:{R_chunks}"
              "User prompt:{user_prompt}"
     )
])

In [ ]:
chain=Retrival_check | llm | parser

In [ ]:
final_response = chain.invoke(
    {
        'user_prompt': user_prompt,
        'R_chunks': R_chunks
    }
)

In [ ]:
final_response

'Based on the user prompt **"Explain Machine Learning"**, here is the evaluation of the 5 retrieved documents:\n\n### **Relevance Score: 40% (2 out of 5 chunks are highly relevant)**\n\n---\n\n### **Detailed Analysis of Each Chunk:**\n\n#### **1. Document 3 (ID: `22b958e2-f5de-4cb7-999b-2b5e10905b7f`)**\n* **Status:** **Highly Relevant (Correct)**\n* **Reason:** This chunk directly explains what Machine Learning is. It defines it as a branch of artificial intelligence focused on enabling computers to learn patterns from data instead of using explicitly programmed rules.\n\n#### **2. Document 4 (ID: `b73365ea-e54f-4251-a509-27b2e9bbe51f`)**\n* **Status:** **Highly Relevant (Correct)**\n* **Reason:** This chunk explains the core mechanism of how Machine Learning works (providing data to an algorithm, allowing it to learn patterns, training a model, and making predictions on new data).\n\n#### **3. Document 1 (ID: `6710bd2b-2fbd-4bd3-ac8a-8238b7779111`)**\n* **Status:** **Not Relevant / M

'Based on the user prompt **"Explain Machine Learning"**, here is the evaluation of the 5 retrieved documents:\n\n### **Relevance Score: 40% (2 out of 5 chunks are highly relevant)**\n\n---\n\n### **Detailed Analysis of Each Chunk:**\n\n#### **1. Document 3 (ID: `22b958e2-f5de-4cb7-999b-2b5e10905b7f`)**\n* **Status:** **Highly Relevant (Correct)**\n* **Reason:** This chunk directly explains what Machine Learning is. It defines it as a branch of artificial intelligence focused on enabling computers to learn patterns from data instead of using explicitly programmed rules.\n\n#### **2. Document 4 (ID: `b73365ea-e54f-4251-a509-27b2e9bbe51f`)**\n* **Status:** **Highly Relevant (Correct)**\n* **Reason:** This chunk explains the core mechanism of how Machine Learning works (providing data to an algorithm, allowing it to learn patterns, training a model, and making predictions on new data).\n\n#### **3. Document 1 (ID: `6710bd2b-2fbd-4bd3-ac8a-8238b7779111`)**\n* **Status:** **Not Relevant / Marginally Relevant**\n* **Reason:** While it mentions that the "basic idea of machine learning is simple" and lists industries where it is used (healthcare, finance, etc.), it does not actually explain *what* Machine Learning is or how it works. \n\n#### **4. Document 2 (ID: `4a1b5442-24ad-44f7-be93-f4948429f23f`)**\n* **Status:** **Not Relevant**\n* **Reason:** This chunk discusses the skills needed for successful ML projects, the rapid evolution of algorithms, and the importance of good data. It does not explain the concept of Machine Learning itself.\n\n#### **5. Document 5 (ID: `d3b0775a-c6a0-40be-bbc7-e6b21d4442fb`)**\n* **Status:** **Not Relevant**\n* **Reason:** This chunk focuses on "Explainable AI" (XAI) and the initial stages of the machine learning lifecycle (understanding business problems and collecting data). It does not define or explain the core concept of Machine Learning.'
